# Import libraries & Setup environment

In [1]:
import pandas as pd
import numpy as np
import gc
from joblib import Parallel, delayed, externals
import multiprocessing
num_cores = multiprocessing.cpu_count() - 1
import dill
import os
import sys
path2 = os.getcwd()
sys.path.append(path2)

from sklearn.model_selection import train_test_split
import scipy.integrate as spi
from scipy.integrate import solve_ivp
from imblearn.over_sampling import SMOTENC

# Define functions

In [2]:
# Define function for the ode function
def deriv(t, y, beta, pi, rho, phi, delta):
    T, R, E, I, V = y

    dT = -beta * T * V - phi * I * T + rho * R
    dR = phi * I * T - rho * R
    dE = beta * T * V - 4 * E
    dI = 4 * E - delta * I
    dV = pi * I - 15 * V

    return [dT, dR, dE, dI, dV]

# Define function for the TREIV model simulation
def ode_TREIV(func, time, step_size = 0.1, tol = 1e-10):
    beta, pi, rho, phi, delta, tau = func

    y0 = [10**7, 0.0, 1.0, 0.0, 1.0]

    t_start = -tau
    t_end = -tau + time
    times = np.arange(t_start, t_end + step_size, step_size)
    times = np.round(times, int(-np.log10(step_size)))

    ode = solve_ivp(
        fun = deriv,
        t_span = (times[0], times[-1]),
        t_eval = times,
        method = 'LSODA',
        y0 = y0,
        args = (beta, pi, rho, phi, delta),
        rtol = tol,
        atol = tol
    )

    vl_sim = np.log10(np.maximum(ode.y[4], tol))

    return pd.DataFrame({
        "Day_since_symp": ode.t,
        "VL_sim": vl_sim
    })

# Define function to simulate VL trajectory row-by-row
def simulate_single_row(row_tuple):
    index, row = row_tuple
    pars = [
        10 ** row['log10beta_mode'],
        10 ** row['log10pi_mode'],
        10 ** row['log10rho_mode'],
        10 ** row['log10phi_mode'],
        row['delta_mode'],
        row['tau_mode']
    ]
    fitted = ode_TREIV(pars, duration)
    fitted['hosp_id'] = int(row['hosp_id'])
    return fitted[['hosp_id', 'Day_since_symp', 'VL_sim']]

# Define function to simulate VL trajectory in a dataframe form
def simulate_trajec(dframe):
    data = dframe.copy()
    data['hosp_id'] = np.arange(1, len(data) + 1)

    rows_list = list(enumerate(data.to_dict(orient = 'records')))

    res = [simulate_single_row(row) for row in rows_list]

    # Collate simulated result for each row
    est_vl = pd.concat(res, ignore_index = True)

    # Setting the boundaries between day 0 and day 20
    est_fil = est_vl[(est_vl['Day_since_symp'] >= -1e-5) & (est_vl['Day_since_symp'] <= 20 + 1e-5)].copy()
    est_fil['Day_since_symp'] = est_fil['Day_since_symp'].round(1)
    
    # Pivot wider
    est_pivot = est_fil.pivot(index='hosp_id', columns='Day_since_symp', values='VL_sim').reset_index()
    
    # Column clean up/merging step
    drop_cols = ['log10beta_mode', 'log10pi_mode', 'log10rho_mode', 'log10phi_mode', 'delta_mode', 'tau_mode']
    df_droped = [c for c in drop_cols if c in data.columns]
    data_output = data.drop(columns = df_droped).merge(est_pivot, on = 'hosp_id', how = 'left')
    
    return data_output

# Define function to mimic/randomly assign diagnostic day
def diag_rand(dframe):
    np.random.seed(123) # Set seed for reproducibility

    data = dframe.copy()
    n_rows = len(data)
    
    # Simulate lognormal numbers follow mean 3.12 and variance 10.36
    ## Setting size to 100k to ensure enough random number assigned after filter nuber on threshold
    random_lognorm = np.random.lognormal(mean = 3.12, sigma = np.sqrt(10.36), size = 150000)
    # Bound thresholds
    random_lognorm = random_lognorm[(random_lognorm >= 0.5) & (random_lognorm <= 6)]
    
    # Downsample and round to nearest 0.5
    random_lognorm_rounded = np.round(random_lognorm[:n_rows] * 2) / 2
    
    temp_df = pd.DataFrame({
        'hosp_id': data['hosp_id'],
        'day_det': random_lognorm_rounded,
        'day_det2': random_lognorm_rounded + 2
    })
    
    # Initialize list to collect VL value
    vl_sim = []
    vl_sim2 = []
    
    for _, row in temp_df.iterrows():
        h_id = row['hosp_id']
        d1 = round(row['day_det'], 1)
        d2 = round(row['day_det2'], 1)
        
        # Pull records from parsed column names matching time stamps
        sub_data = data[data['hosp_id'] == h_id]
        
        v1 = sub_data[d1].values[0] if d1 in sub_data.columns else np.nan
        v2 = sub_data[d2].values[0] if d2 in sub_data.columns else np.nan
        
        vl_sim.append(v1)
        vl_sim2.append(v2)
        
    temp_df['vl_sim'] = vl_sim
    temp_df['vl_sim2'] = vl_sim2
    
    # Final Join step
    data_f = data.merge(temp_df[['hosp_id', 'vl_sim', 'day_det', 'vl_sim2', 'day_det2']], on = 'hosp_id', how = 'left')
    return data_f

# Define function to mimic/randomly assign diagnostic day
def diag_rand_train(dframe):
    np.random.seed(123) # Set seed for reproducibility

    data = dframe.copy()
    n_rows = len(data)
    
    # Simulate lognormal numbers follow mean 3.12 and variance 10.36
    ## Setting size to 100k to ensure enough random number assigned after filter nuber on threshold
    random_lognorm = np.random.lognormal(mean = 3.12, sigma = np.sqrt(10.36), size = 150000)
    # Bound thresholds
    random_lognorm = random_lognorm[(random_lognorm >= 0.5) & (random_lognorm <= 6)]
    
    # Downsample and round to nearest 0.5
    random_lognorm_rounded = np.round(random_lognorm[:n_rows] * 2) / 2
    
    temp_df = pd.DataFrame({
        'hosp_id': data['hosp_id'],
        'day_det': random_lognorm_rounded,
        'day_det2': random_lognorm_rounded + 2
    })
    #temp_df = temp_df.drop_duplicates(subset = "hosp_id")
    
    # Initialize list to collect VL value
    vl_sim = []
    vl_sim2 = []
    
    for _, row in temp_df.iterrows():
        h_id = row['hosp_id']
        d1 = str(round(row['day_det'], 1))
        d2 = str(round(row['day_det2'], 1))
        
        # Pull records from parsed column names matching time stamps
        sub_data = data[data['hosp_id'] == h_id]
        
        v1 = sub_data[d1].values[0] if d1 in sub_data.columns else np.nan
        v2 = sub_data[d2].values[0] if d2 in sub_data.columns else np.nan
        
        vl_sim.append(v1)
        vl_sim2.append(v2)
        
    temp_df['vl_sim'] = vl_sim
    temp_df['vl_sim2'] = vl_sim2
    
    # Final Join step
    data_f = data.merge(temp_df[['hosp_id', 'vl_sim', 'day_det', 'vl_sim2', 'day_det2']], on = 'hosp_id', how = 'left')
    return data_f

# Define function for data splitting and SMOTE oversampling process
def split_process_valid_SMOTE_first(dframe, seed_val):
    # Splitting data into training, validation and test sets
    train_df, temp_df = train_test_split(dframe, train_size = 0.6, stratify = dframe['outcome'], random_state = seed_val)
    valid_df, test_df = train_test_split(temp_df, train_size = 0.5, stratify = temp_df['outcome'], random_state = seed_val)

    train_df = simulate_trajec(train_df)
    valid_sim = simulate_trajec(valid_df)
    test_sim = simulate_trajec(test_df)

    train_df.columns = train_df.columns.astype(str)

    cat_cols = ['age', 'gender', 'vaccination', 'comorbidity']
    train_x = train_df.drop(columns = ['hosp_id', 'outcome'])
    train_y = train_df['outcome']

    # Perform SMOTE oversampling
    sm = SMOTENC(
        categorical_features = cat_cols,
        k_neighbors = 2,
        random_state = 123,
        sampling_strategy = 'auto')
    x_res, y_res = sm.fit_resample(train_x, train_y)

    train_res = pd.DataFrame(x_res, columns = train_x.columns)
    train_res['outcome'] = y_res

    for col in cat_cols:
        train_res[col] = train_res[col].astype('category')
    
    train_res['age'] = train_res['age'].cat.reorder_categories(["18-59", "60-79", ">=80"], ordered = True)
    train_res['gender'] = train_res['gender'].cat.reorder_categories(["Male", "Female"], ordered = True)
    train_res['vaccination'] = train_res['vaccination'].cat.reorder_categories(["Notfull", "Full", "Booster"], ordered = True)
    train_res['comorbidity'] = train_res['comorbidity'].cat.reorder_categories(["0", "1-2", ">=3"], ordered = True)
    train_res['outcome'] = train_res['outcome'].cat.reorder_categories(["Non severe", "Severe"], ordered = True)

    if '-0.0' in train_res.columns:
        train_res = train_res.rename(columns = {'-0.0': '0.0'})
    
    train_res['hosp_id'] = range(1, len(train_res) + 1)

    return {
        'train': train_res,
        'valid': valid_sim,
        'test': test_sim
    }

# Define function to getting the dataframe of the day of diagnosis assigned to each individual
def diag_list_func(split_list):
    diag_list = []

    for i in range(100):
        split = split_list[i]

        dfs = []

        for set_name in ["train", "valid", "test"]:
            df = split[set_name]

            temp = pd.DataFrame({
                "run": i,
                "set": set_name,
                "hosp_id": df["hosp_id"].to_numpy(),
                "diag": df["day_det"].to_numpy()
            })

            dfs.append(temp)

        diag_df = pd.concat(dfs, ignore_index = True)

        diag_list.append(diag_df)

    temp = pd.concat(diag_list, ignore_index = True)
    
    for i in range(11):
        temp[f"diag_p{i}"] = temp['diag'] + i

    return temp

# Define function to get the sequential viral load at diagnosis
def seq_fetch_func(i, frame, diag_list, set_type):
    data = frame.copy()

    temp_diag = diag_list[(diag_list["run"] == i) & (diag_list["set"] == set_type)].copy()

    vl_lists = [[] for _ in range(11)]

    for _, row in temp_diag.iterrows():
        h_id = row["hosp_id"]
        sub_data = data[data["hosp_id"] == h_id]

        for j in range(11):
            diag_col = row[f"diag_p{j}"]

            if set_type == 'train':
                if len(sub_data) > 0 and str(diag_col) in sub_data.columns:
                    value = sub_data[str(diag_col)].iloc[0]
                else:
                    value = np.nan
                vl_lists[j].append(value)
            else:
                if len(sub_data) > 0 and diag_col in sub_data.columns:
                    value = sub_data[diag_col].iloc[0]
                else:
                    value = np.nan
                vl_lists[j].append(value)

    for j in range(11):
        temp_diag[f"vl_diag_{j}"] = vl_lists[j]

    vl_cols = [f"vl_diag_{j}" for j in range(11)]

    data_df = data[['hosp_id', 'age', 'gender', 'vaccination', 'comorbidity', 'VL', 'outcome']].copy()

    data_f = data_df.merge(temp_diag[["hosp_id"] + vl_cols], on = "hosp_id", how = "left")

    return data_f

# Import dataset into environment

In [ ]:
path = '~path_to_dataset'

# Import dataset into environment
df = pd.read_csv(f"{path}/cluster_uniqID_symptom.csv")
# Select columns to be used in following analysis
df_tab = df[['age_cat', 'sex', 'vaccination', 'comorbidity', 'VL', 'log10beta_mode', 'log10pi_mode', 'log10rho_mode', 'log10phi_mode', 'delta_mode', 'tau_mode', 'outcome']].copy()
# Rename columns
df_tab = df_tab.rename(columns = {'age_cat': 'age', 'sex': 'gender'})

# Fixing column types
cat_cols = ['age', 'gender', 'vaccination', 'comorbidity', 'outcome']
for col in cat_cols:
    df_tab[col] = df_tab[col].astype('category')

# Split and process data

In [4]:
duration = 60.0 # Define duration period to be simulated
if __name__ == '__main__':
    try:
        # Run the entire split + simulation concurrently across 100 iterations
        split_list_valid_smote = Parallel(n_jobs = num_cores, verbose = 10)(
            delayed(split_process_valid_SMOTE_first)(df_tab, seed) for seed in range(100)
        )
    finally:
        # Hard shutdown to purge worker memory pools instantly
        externals.loky.get_reusable_executor().shutdown(wait = True)
        gc.collect()

split_list_valid_smote_final = []
for split in split_list_valid_smote:
    train_final = diag_rand_train(split['train'])
    valid_final = diag_rand(split['valid'])
    test_final  = diag_rand(split['test'])
    
    split_list_valid_smote_final.append({
        'train': train_final,
        'valid': valid_final,
        'test': test_final
    })

[Parallel(n_jobs=19)]: Using backend LokyBackend with 19 concurrent workers.
[Parallel(n_jobs=19)]: Done   3 tasks      | elapsed:   26.6s
[Parallel(n_jobs=19)]: Done  12 tasks      | elapsed:   27.6s
[Parallel(n_jobs=19)]: Done  23 tasks      | elapsed:   53.5s
[Parallel(n_jobs=19)]: Done  34 tasks      | elapsed:   55.0s
[Parallel(n_jobs=19)]: Done  47 tasks      | elapsed:  1.4min
[Parallel(n_jobs=19)]: Done  60 tasks      | elapsed:  1.8min
[Parallel(n_jobs=19)]: Done  74 out of 100 | elapsed:  1.8min remaining:   38.5s
[Parallel(n_jobs=19)]: Done  85 out of 100 | elapsed:  2.3min remaining:   23.9s
[Parallel(n_jobs=19)]: Done  96 out of 100 | elapsed:  2.6min remaining:    6.3s
[Parallel(n_jobs=19)]: Done 100 out of 100 | elapsed:  2.6min finished


In [5]:
# Getting data without any VL information
fea_without_vl = ['age', 'gender', 'vaccination', 'comorbidity', 'outcome']

split_list_fil_valid_smote_final = [
    {
        "train": split["train"][fea_without_vl].copy(),
        "valid": split["valid"][fea_without_vl].copy(),
        "test": split["test"][fea_without_vl].copy()
    }
    for split in split_list_valid_smote_final
]

# Getting data with simulated viral load at symptom onset
split_list_sim_onset_valid_smote_final = []
for split in split_list_valid_smote_final:
    fea_simonset = ['age', 'gender', 'vaccination', 'comorbidity', '0.0', 'outcome']
    fea_simonset2 = ['age', 'gender', 'vaccination', 'comorbidity', 0.0, 'outcome']
    split_list_sim_onset_valid_smote_final.append({
        "train": split["train"][fea_simonset].rename(columns = {'0.0': "vl_sim_onset"}).copy(),
        "valid": split["valid"][fea_simonset2].rename(columns = {0.0: "vl_sim_onset"}).copy(),
        "test": split["test"][fea_simonset2].rename(columns = {0.0: "vl_sim_onset"}).copy()
        })

# Getting data with VL at diagnosis
fea_vldiag = ['age', 'gender', 'vaccination', 'comorbidity', 'vl_sim', 'outcome']

split_list_vldiag_valid_smote_final = [
    {
        "train": split["train"][fea_vldiag].copy(),
        "valid": split["valid"][fea_vldiag].copy(),
        "test": split["test"][fea_vldiag].copy()
    }
    for split in split_list_valid_smote_final
]

# Getting data with VL at diagnosis and 2 days after
fea_vldiag2 = ['age', 'gender', 'vaccination', 'comorbidity', 'vl_sim', 'vl_sim2', 'outcome']

split_list_diag2_valid_smote_final = [
    {
        "train": split["train"][fea_vldiag2].copy(),
        "valid": split["valid"][fea_vldiag2].copy(),
        "test": split["test"][fea_vldiag2].copy()
    }
    for split in split_list_valid_smote_final
]

In [6]:
# Getting data with VL at diagnosis and days following
diag_list_df = diag_list_func(split_list_valid_smote_final)
split_seq_list = []
for i in range(len(split_list_valid_smote)):
    train_df = seq_fetch_func(i, split_list_valid_smote[i]['train'], diag_list_df, 'train')
    valid_df = seq_fetch_func(i, split_list_valid_smote[i]['valid'], diag_list_df, 'valid')
    test_df = seq_fetch_func(i, split_list_valid_smote[i]['test'], diag_list_df, 'test')

    split_seq_list.append({
        'train': train_df,
        'valid': valid_df,
        'test': test_df
    })

In [7]:
# Getting data with VL at diagnosis and 1 day after
fea_vldiag1 = ['age', 'gender', 'vaccination', 'comorbidity', 'vl_diag_0', 'vl_diag_1', 'outcome']

split_list_diag1_valid_smote_final = [
    {
        "train": split["train"][fea_vldiag1].rename(columns = {"vl_diag_0": "vl_sim", "vl_diag_1": "vl_sim3"}).copy(),
        "valid": split["valid"][fea_vldiag1].rename(columns = {"vl_diag_0": "vl_sim", "vl_diag_1": "vl_sim3"}).copy(),
        "test": split["test"][fea_vldiag1].rename(columns = {"vl_diag_0": "vl_sim", "vl_diag_1": "vl_sim3"}).copy()
    }
    for split in split_seq_list
]

# Save data processed

In [ ]:
path2 = os.getcwd()

state = {
    "split_list_valid_smote": split_list_valid_smote,
    "split_list_valid_smote_final": split_list_valid_smote_final,
    "split_list_fil_valid_smote_final": split_list_fil_valid_smote_final, # Data list without VL information.
    "split_list_sim_onset_valid_smote_final": split_list_sim_onset_valid_smote_final, # Data list with simulated VL at symptom onset.
    "split_list_vldiag_valid_smote_final": split_list_vldiag_valid_smote_final, # Data list with VL at diagnosis.
    "split_list_diag1_valid_smote_final": split_list_diag1_valid_smote_final, # Data list with VL at diagnosis & VL at 1-day after diagnosis.
    "split_list_diag2_valid_smote_final": split_list_diag2_valid_smote_final, # Data list with VL at diagnosis & VL at 2-days after diagnosis.
    "split_seq_list": split_seq_list
}

save_file = os.path.join(path2, "session.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

print(f"Saved to: {save_file}")